# P-003, one cell

Order cannot break a single cell. Training then both held-out sixties.


In [ ]:
%%bash
set -e
cd /content
rm -rf battery
git clone -q https://github.com/narko4u/agent-transition-battery battery
cd battery
pip install -q transformers peft datasets bitsandbytes accelerate
nvidia-smi --query-gpu=name,memory.total --format=csv
python tools/train_rlvr.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --tasks 200 --samples 4 --epochs 3 --max-new 320 --out /content/rlvr-1.5b-e3
python tools/eval_local.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --family families/mbpp-hold60 --adapter /content/rlvr-1.5b-e3 --protocol P-003 --out /content/e3-hold60.json
python tools/eval_local.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --family families/mbpp-test60 --adapter /content/rlvr-1.5b-e3 --protocol P-003 --out /content/e3-test60.json
python - <<'EOF'
import json
for name in ("e3-hold60", "e3-test60"):
    d = json.load(open("/content/%s.json" % name))
    print(name.upper(), d["base"]["passed"], d["tuned"]["passed"], "up", d["improved"], "dn", d["regressed"], "delta", d["delta"])
EOF
